# 07 — Clonal fitness across three cohorts

**The reframe.** Progression from CHIP to myeloid neoplasm happens to about
0.5–1% of carriers per year, so a binary "will progress" label discards almost
every participant. Clonal fitness — the exponential growth rate of a mutant
clone —

$$s = \frac{d \log \mathrm{VAF}}{dt}$$

is measurable in **every** carrier, ill or not. Same cohort, orders of magnitude
more usable signal, and it is the mechanism rather than a proxy: an expanding
clone is a clone on its way to becoming a disease.

**Three public sources, pooled.**

| folder | source | participants | timing |
|---|---|---|---|
| `lbc_gse178936` | Robertson 2022, Nat Med | 85 | 4–5 waves, ~3 yr apart |
| `gse288742` | 2025 release, LBC cohort | 63 | 4 waves, no participant overlap |
| `sardinia` | Fabre 2022, Nature Suppl. Table 2 | 349 | up to 5 samplings, median 12.9 yr, **ages recorded** |

SardiNIA is not merely bigger. It records **age at each sampling**, so time
between points is real rather than the "3 years per wave" approximation LBC
forces; and it reaches back to age 55 where LBC starts at 70, covering a
different part of an age-dependent curve.

**What this notebook decides.** Whether the fitness estimates are trustworthy,
how many variants survive to be modelled, and — only if the count clears the
power calculation — whether ESM2 predicts fitness better than knowing the gene.

In [2]:
# ============================================================
# 07_fitness.ipynb  (rewritten)
# Clonal fitness + BioVU + ESM-2 delta → fitness regression
# ============================================================
import os, sys, subprocess, gzip, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    subprocess.run("pip install -q transformers accelerate openpyxl joblib".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
for m in [k for k in list(sys.modules) if k.startswith("src")]:
    del sys.modules[m]

import numpy as np
import pandas as pd
import json, yaml
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
import joblib

pd.set_option("display.width", 170)
CFG = yaml.safe_load(open("configs/config.yaml"))
DATA = Path("data/processed")
DATA.mkdir(parents=True, exist_ok=True)
print("working dir:", os.getcwd())

Mounted at /content/drive
working dir: /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection


## 1. Load all three

Note two different loaders and two different fitness estimators. LBC files are
TSV with a `wave` column; SardiNIA is an Excel sheet with the header on row 4
and an `Age` column. `estimate_fitness_by_age` regresses on real elapsed time —
use it wherever an age or date column exists, because unequal gaps otherwise
bias every slope.

In [ ]:
from src.fitness import (
    load_many, load_sardinia, estimate_fitness,
    estimate_fitness_by_age, fitness_report,
    standardise_within_cohort, missense_targets
)
from src.mutation_utils import parse_protein_change, mutation_key

lbc_ns, lbc_sy = load_many(["data/raw/lbc_gse178936", "data/raw/gse288742"])
sar_ns, sar_sy = load_sardinia("data/raw/sardinia/41586_2022_4785_MOESM5_ESM.xlsx")

print(f"LBC participants: {lbc_ns.participant_id.nunique()}  |  SardiNIA: {sar_ns.participant_id.nunique()}")

fits = {
    "LBC":      estimate_fitness(lbc_ns),
    "SardiNIA": estimate_fitness_by_age(sar_ns, min_af=0.005),
}
syns = {
    "LBC":      estimate_fitness(lbc_sy),
    "SardiNIA": estimate_fitness_by_age(sar_sy, min_af=0.005),
}

print("=" * 60); print("LBC"); print("=" * 60)
_ = fitness_report(fits["LBC"], syns["LBC"], min_traj=5)

print("=" * 60); print("SardiNIA"); print("=" * 60)
_ = fitness_report(fits["SardiNIA"], syns["SardiNIA"], min_traj=5)

ns_fit = standardise_within_cohort(fits, syns)
print("ns_fit shape:", ns_fit.shape)

[lbc] non-synonymous: 10,173 rows, 85 participants
[lbc] synonymous: 3,544 rows, 84 participants
[lbc] non-synonymous: 3,616 rows, 63 participants
[lbc] synonymous: 873 rows, 63 participants
[pool] 13,789 non-syn / 4,417 syn rows, 148 participants
[sardinia] 3,886 non-syn / 406 syn rows, 349 participants, age 55-103
LBC participants: 148  |  SardiNIA: 335
[fitness] 174 trajectories (VAF>=0.01, DP>=500, >=3 waves)
[fitness/age] 417 trajectories (VAF>=0.005, >=3 timepoints, median span 10.3y)
[fitness] 99 trajectories (VAF>=0.01, DP>=500, >=3 waves)
[fitness/age] 22 trajectories (VAF>=0.005, >=3 timepoints, median span 10.2y)
LBC
FITNESS CALIBRATION
[noise] synonymous n=99, mean=-0.0045 (should be ~0), sd=0.0288, p95=+0.0356

[1] Non-synonymous above the synonymous 95th percentile: 21.8%  (5% = chance)
    ok: 4.4x enrichment over chance.

[2] Canonical CHIP drivers — these must surface if the estimates are real:
        count    mean
gene                 
ASXL1       2  0.0982
TP53     

`load_many` warns if a participant appears in more than one release. It should
print nothing — GSE178936 uses `CHIP_LBC*` identifiers and GSE288742 uses
`CHIP2024_*`, with zero overlap.

Variants are keyed on **genomic coordinates**, not protein notation. About 15%
of rows (splice, UTR) have no `protein_substitution`, and how a missing value
behaves in string concatenation is pandas-version dependent — newer pandas keeps
NA and `groupby` silently drops those trajectories, older pandas coerces to the
literal `"nan"` and merges every distinct variant in a gene into one fake
trajectory. Both are wrong and they give different totals on the same file.

## 2. Fitness per cohort, calibrated separately

In [ ]:
targets = missense_targets(ns_fit)

parsed = targets["protein_substitution"].apply(parse_protein_change)
mask = parsed.notnull()
targets = targets[mask].copy()
parsed = parsed[mask]

targets["aa_wt"]  = [p["wt"] for p in parsed]
targets["aa_pos"] = [p["pos"] for p in parsed]
targets["aa_mut"] = [p["mut"] for p in parsed]
targets["mut_key"] = [
    mutation_key(g, p)
    for g, p in zip(targets["gene"], targets["protein_substitution"])
]
targets = targets[
    targets["aa_mut"].notnull() & (targets["aa_mut"] != targets["aa_wt"])
].copy()
targets["cohort"] = "LBC_SardiNIA"

print(f"LBC/SardiNIA unique missense: {len(targets)}")
print(targets[["gene", "protein_substitution", "s_z", "n_obs"]].head())

[targets] 365 missense trajectories -> 275 unique protein variants
  at n=275, 80% power to detect r >= 0.168
LBC/SardiNIA unique missense: 273
    gene protein_substitution       s_z  n_obs
0  ASXL1              p.M341L  2.381096      1
1  ASXL1              p.Q803R  0.333699      1
2   BCOR             p.D1502N -0.742756      1
3   BCOR              p.D782N  0.054308      1
4   BCOR              p.P707L  2.429376      1


### Reading the two reports

**LBC** should show ~3.8× enrichment over its synonymous null, with TET2, JAK2
and DNMT3A at the top at 0.04–0.06 per year.

**SardiNIA** should rank SRSF2 > SF3B1 > ASXL1 > TET2 > DNMT3A — the exact
ordering Fabre reported on this data, spliceosome factors fastest and DNMT3A
slowest. That agreement is the only external validation available and it matters
more than any single number here.

**One honest problem.** SardiNIA's synonymous slopes are not centred on zero at
any VAF floor, and only ~22 survive filtering. Its published table lists only
variants that were *detected*, and a growing clone is likelier to be detected —
so the synonymous set is itself enriched for growth. The z-scores below correct
the offset; they do not undo the ascertainment. State this in any write-up.

In [ ]:
# Chỉnh path nếu file nằm chỗ khác trên Drive
BIOVU_RAW = Path("data/raw/Mack_Pershad_s01.txt")
if not BIOVU_RAW.exists():
    BIOVU_RAW = Path("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection/data/raw/Mack_Pershad_s01.txt")

df = pd.read_csv(BIOVU_RAW, sep="\t")
biovu = df[df["Mutation_Type"] == "nonsynonymous SNV"].copy()
biovu = biovu.dropna(subset=["Chr", "Start", "Ref", "Alt", "VAF1", "VAF2", "deltaT(years)"])
biovu = biovu[biovu["deltaT(years)"] > 0].copy()

VAF_FLOOR = 0.001
biovu["VAF1"] = biovu["VAF1"].clip(lower=VAF_FLOOR)
biovu["VAF2"] = biovu["VAF2"].clip(lower=VAF_FLOOR)
biovu["s"] = (np.log(biovu["VAF2"]) - np.log(biovu["VAF1"])) / biovu["deltaT(years)"]
biovu["mean_af"] = (biovu["VAF1"] + biovu["VAF2"]) / 2
biovu["n_obs"] = 1
biovu["cohort"] = "BioVU"
biovu["chrom"] = biovu["Chr"].astype(str).str.replace("^chr", "", regex=True)
biovu["key"] = (
    biovu["chrom"] + "_" + biovu["Start"].astype(str) + "_" +
    biovu["Ref"].astype(str) + "_" + biovu["Alt"].astype(str)
)
print(f"BioVU missense rows: {len(biovu)}")
print(biovu["s"].describe().round(3))

BioVU missense rows: 462
count    462.000
mean       0.091
std        0.247
min       -1.461
25%        0.000
50%        0.069
75%        0.155
max        2.703
Name: s, dtype: float64


In [ ]:
VEP_PATH = Path("data/processed/biovu_vep_input_VEP.vcf.gz")
if not VEP_PATH.exists():
    VEP_PATH = Path("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection/data/processed/biovu_vep_input_VEP.vcf.gz")

rows = []
with gzip.open(VEP_PATH, "rt") as f:
    for line in f:
        if line.startswith("##"):
            continue
        if line.startswith("#CHROM"):
            header = line.strip().lstrip("#").split("\t")
            continue
        rows.append(line.strip().split("\t"))

vep = pd.DataFrame(rows, columns=header)

def parse_csq(info):
    if "CSQ=" not in info:
        return None, None, None, None
    csq = info.split("CSQ=")[1].split(";")[0]
    for transcript in csq.split(","):
        parts = transcript.split("|")
        if len(parts) < 16:
            continue
        consequence, gene, biotype = parts[1], parts[3], parts[7]
        prot_pos, aa = parts[14], parts[15]
        if "missense_variant" in consequence and biotype == "protein_coding" and aa and "/" in aa and prot_pos:
            return gene, consequence, prot_pos, aa
    return None, None, None, None

parsed = vep["INFO"].apply(parse_csq)
vep["Gene"] = [x[0] for x in parsed]
vep["Consequence"] = [x[1] for x in parsed]
vep["Protein_position"] = [x[2] for x in parsed]
vep["Amino_acids"] = [x[3] for x in parsed]

def make_subst(row):
    if pd.isna(row["Amino_acids"]) or pd.isna(row["Protein_position"]):
        return None
    wt, mut = row["Amino_acids"].split("/")
    return f"{wt}{row['Protein_position']}{mut}"

vep["protein_substitution"] = vep.apply(make_subst, axis=1)
vep_missense = vep[vep["protein_substitution"].notna()].copy()
vep_missense["key"] = (
    vep_missense["CHROM"].astype(str) + "_" +
    vep_missense["POS"].astype(str) + "_" +
    vep_missense["REF"].astype(str) + "_" +
    vep_missense["ALT"].astype(str)
)
vep_slim = vep_missense[["key", "Gene", "protein_substitution", "Protein_position", "Amino_acids"]].drop_duplicates("key")
print(f"VEP missense with protein change: {len(vep_slim)}")

VEP missense with protein change: 279


## 4. How many variants can be modelled?

A variant seen in three people gives three noisy estimates of one underlying
quantity — average them. The count of **unique protein variants**, not
trajectories and not participants, is the sample size for a regression.

`missense_targets` reports the effect size detectable at that n. Read it
**before** running the model, not after: it is the only way to tell an honest
negative result from an underpowered one.

In [ ]:
# BioVU: raw two-point slope only. Do NOT copy s into s_z.
# BioVU has no synonymous trajectories in Mack_Pershad_s01.txt, so there is
# no noise floor to standardise against. Leave s_z empty.

merged = biovu.merge(vep_slim, on="key", how="inner")
if "Gene_y" in merged.columns:
    merged["Gene"] = merged["Gene_y"]
elif "Gene_x" in merged.columns:
    merged["Gene"] = merged["Gene_x"]

merged["gene"] = merged["Gene"]
merged["aa_wt"] = merged["Amino_acids"].str.split("/").str[0]
merged["aa_mut"] = merged["Amino_acids"].str.split("/").str[1]
merged["aa_pos"] = pd.to_numeric(merged["Protein_position"], errors="coerce")
merged["s_z"] = np.nan

biovu_targets = merged[[
    "gene", "protein_substitution", "aa_wt", "aa_pos", "aa_mut",
    "s", "s_z", "mean_af", "n_obs", "cohort", "key"
]].dropna(subset=["aa_pos", "aa_wt", "aa_mut"]).copy()
biovu_targets["aa_pos"] = biovu_targets["aa_pos"].astype(int)
biovu_targets["mut_key"] = [
    mutation_key(g, p)
    for g, p in zip(biovu_targets["gene"], biovu_targets["protein_substitution"])
]

# collapse people-level rows → one row per canonical mutation
biovu_unique = (
    biovu_targets.groupby("mut_key", as_index=False)
    .agg(
        gene=("gene", "first"),
        protein_substitution=("protein_substitution", "first"),
        aa_wt=("aa_wt", "first"),
        aa_pos=("aa_pos", "first"),
        aa_mut=("aa_mut", "first"),
        s=("s", "median"),
        mean_af=("mean_af", "median"),
        n_obs=("n_obs", "sum"),
        cohort=("cohort", "first"),
    )
)
biovu_unique["s_z"] = np.nan

print(f"BioVU person-rows: {len(biovu_targets)}")
print(f"BioVU unique mut_key: {len(biovu_unique)}")
print("BioVU s (raw only):")
print(biovu_unique["s"].describe().round(4))
biovu_targets.to_csv(DATA / "biovu_missense_targets.csv", index=False)
biovu_unique.to_csv(DATA / "biovu_unique_targets.csv", index=False)


BioVU person-rows: 462
BioVU unique mut_key: 275
BioVU s (raw only):
count    275.0000
mean       0.0905
std        0.2403
min       -1.4613
25%        0.0028
50%        0.0698
75%        0.1650
max        1.4403
Name: s, dtype: float64


## 5. Deduplicate on `mut_key` — and do **not** mix scales

Two separate bugs lived in the old concat:

1. Group-by included `protein_substitution`, so `p.R882H` and `R882H` stayed as two rows even when `mut_key` was identical. Those copies leak across CV folds.
2. BioVU `s_z` was set equal to raw `s` (std ~0.24). LBC/SardiNIA `s_z` is a within-cohort z (range about −2.7 to +7.9). Concatenating them trains on a 10× scale artefact.

BioVU has **no synonymous** rows in Mack & Pershad, so it cannot be z-scored the same way. Keep it as a raw-`s` holdout. Train only on LBC+SardiNIA `s_z`.


In [ ]:
# LBC/SardiNIA: one row per canonical mutation (ignore HGVS spelling)
lbc_unique = (
    targets.groupby("mut_key", as_index=False)
    .agg(
        gene=("gene", "first"),
        protein_substitution=("protein_substitution", "first"),
        aa_wt=("aa_wt", "first"),
        aa_pos=("aa_pos", "first"),
        aa_mut=("aa_mut", "first"),
        s=("s", "median"),
        s_z=("s_z", "median"),
        n_obs=("n_obs", "sum"),
        n_rows=("s", "size"),
    )
)
lbc_unique["cohort"] = "LBC_SardiNIA"

print(f"LBC/SardiNIA person-like rows: {len(targets)}")
print(f"LBC/SardiNIA unique mut_key:   {len(lbc_unique)}")
print(lbc_unique["s_z"].describe().round(4))

# Overlap diagnostic — do not pool these into one target
overlap_keys = sorted(set(lbc_unique["mut_key"]) & set(biovu_unique["mut_key"]))
print(f"\nOverlap mut_key (LBC ∩ BioVU): {len(overlap_keys)}")

L = lbc_unique.set_index("mut_key")
B = biovu_unique.set_index("mut_key")
pivot = pd.DataFrame({
    "mut_key": overlap_keys,
    "gene": [L.loc[k, "gene"] for k in overlap_keys],
    "lbc_s": [L.loc[k, "s"] for k in overlap_keys],
    "lbc_s_z": [L.loc[k, "s_z"] for k in overlap_keys],
    "biovu_s": [B.loc[k, "s"] for k in overlap_keys],
})
pivot["same_sign"] = np.sign(pivot["lbc_s"]) == np.sign(pivot["biovu_s"])
pivot["delta_s"] = pivot["biovu_s"] - pivot["lbc_s"]
pivot = pivot.sort_values("delta_s", key=lambda x: x.abs(), ascending=False)

print(f"same sign: {pivot['same_sign'].mean():.1%} ({int(pivot['same_sign'].sum())}/{len(pivot)})")
if len(pivot) >= 3:
    print(f"Pearson  s     : {pivot['lbc_s'].corr(pivot['biovu_s']):.3f}")
    print(f"Spearman s     : {pivot['lbc_s'].corr(pivot['biovu_s'], method='spearman'):.3f}")
    print(f"Pearson  s_z~s : {pivot['lbc_s_z'].corr(pivot['biovu_s']):.3f}")
display(pivot.head(15) if 'display' in dir() else pivot.head(15))

# TRAIN set = LBC/SardiNIA only
pooled_unique = lbc_unique.copy()
print(f"\nTrain variants (LBC/SardiNIA only): {len(pooled_unique)}")

lbc_unique.to_csv(DATA / "lbc_sardinia_unique_targets.csv", index=False)
pivot.to_csv(DATA / "overlap53_lbc_vs_biovu.csv", index=False)
# keep a file named pooled_* but it is LBC-only and documented
pooled_unique.to_csv(DATA / "pooled_missense_targets.csv", index=False)
print("Saved unique LBC targets + overlap pivot. BioVU is NOT in the train table.")


LBC/SardiNIA person-like rows: 273
LBC/SardiNIA unique mut_key:   266
count    266.0000
mean       0.6126
std        1.2973
min       -1.8106
25%       -0.1690
50%        0.3779
75%        1.0377
max        7.9285
Name: s_z, dtype: float64

Overlap mut_key (LBC ∩ BioVU): 53
same sign: 71.7% (38/53)
Pearson  s     : 0.225
Spearman s     : 0.210
Pearson  s_z~s : 0.126


,mut_key,gene,lbc_s,lbc_s_z,biovu_s,same_sign,delta_s
37,KRAS_T58I,KRAS,0.072525,0.832998,-0.263656,False,-0.336180
12,DNMT3A_L754H,DNMT3A,0.038521,0.365415,0.327709,True,0.289188
45,SRSF2_P95A,SRSF2,0.090146,1.075303,0.371499,True,0.281353
14,DNMT3A_M880V,DNMT3A,-0.008894,-0.286585,-0.236969,True,-0.228075
41,SF3B1_H662Q,SF3B1,0.004168,-0.106967,0.225475,True,0.221307
46,SRSF2_P95H,SRSF2,0.125843,1.566177,0.335784,True,0.209941
5,DNMT3A_F755S,DNMT3A,-0.033915,-1.326687,0.174109,False,0.208024
48,TET2_C1135Y,TET2,-0.004457,-0.225571,0.200324,False,0.204781
27,DNMT3A_R882S,DNMT3A,0.099835,1.208531,0.303938,True,0.204103
44,SF3B1_R625L,SF3B1,0.084595,0.998974,0.267276,True,0.182681



Train variants (LBC/SardiNIA only): 266
Saved unique LBC targets + overlap pivot. BioVU is NOT in the train table.


## 6. ESM-2 → fitness — LBC/SardiNIA only

Target is **`s_z` on LBC+SardiNIA unique `mut_key`**. BioVU stays out until it has a synonymous noise floor or an independently justified scale.

Do not print ΔR² between two negative CV scores. If every fold is worse than predicting the mean, the comparison is not a result.


In [ ]:
from src.embedding import ESM2Embedder, embed_mutations
from src.sequences import get_sequences

mut_table = pooled_unique.copy()
mut_table["kind"] = "missense"

sequences = get_sequences(mut_table["gene"].unique().tolist())
print(f"Sequences loaded for {len(sequences)} genes")

embedder = ESM2Embedder()
emb = embed_mutations(mut_table, sequences, embedder, batch_size=4)

pooled_emb = pooled_unique.merge(emb[["mut_key", "delta"]], on="mut_key", how="inner")
print(f"Train variants with ESM-2 delta: {len(pooled_emb)}  (should be ≤ {len(pooled_unique)})")
print("cohorts in train:", pooled_emb.get("cohort", pd.Series(dtype=str)).value_counts().to_dict() if "cohort" in pooled_emb.columns else "LBC_SardiNIA")
pooled_emb.to_pickle(DATA / "lbc_sardinia_with_delta.pkl")


Sequences loaded for 52 genes
[bionemo-esm2] loading nvidia/esm2_t33_650M_UR50D on cuda torch.float16 ...


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

The repository nvidia/esm2_t33_650M_UR50D contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/nvidia/esm2_t33_650M_UR50D .
 You can inspect the repository content at https://hf.co/nvidia/esm2_t33_650M_UR50D.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


esm_nv.py:   0%|          | 0.00/38.6k [00:00<?, ?B/s]

[transformers] Encountered exception while importing transformer_engine: No module named 'transformer_engine'


[bionemo-esm2] nvidia/esm2_t33_650M_UR50D failed: This modeling file requires the following packages that were not found in your environment: transformer_engine. Run `pip install transformer_engine`
[bionemo-esm2] loading facebook/esm2_t33_650M_UR50D on cuda torch.float16 ...


... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[bionemo-esm2] ready  backend=facebook_hf  model=facebook/esm2_t33_650M_UR50D  hidden dim=1280  alloc=1302MB reserved=1428MB
[embed] 266 rows -> 266 unique missense mutations
[embed] mapped 260/266 (reference mismatches: 34, 13.1%)
  [bionemo-esm2] 1/260  alloc=1312MB reserved=1504MB
  [bionemo-esm2] 21/260  alloc=1312MB reserved=1510MB
  [bionemo-esm2] 41/260  alloc=1312MB reserved=1506MB
  [bionemo-esm2] 61/260  alloc=1312MB reserved=1506MB
  [bionemo-esm2] 81/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 101/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 121/260  alloc=1312MB reserved=1493MB
  [bionemo-esm2] 141/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 161/260  alloc=1312MB reserved=1481MB
  [bionemo-esm2] 181/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 201/260  alloc=1312MB reserved=1508MB
  [bionemo-esm2] 221/260  alloc=1312MB reserved=1508MB
  [bionemo-esm2] 241/260  alloc=1312MB reserved=1512MB
  [bionemo-esm2] 1/260  alloc=1312MB reserved=1504MB
  [bio

In [ ]:
# Train ONLY on LBC/SardiNIA unique variants. Target = s_z.
X_delta = np.vstack(pooled_emb["delta"].values)
y = pooled_emb["s_z"].astype(float).values
genes = pooled_emb[["gene"]]

print("y = s_z  mean={:.3f}  std={:.3f}  n={}".format(np.nanmean(y), np.nanstd(y), y.size))
assert np.isfinite(y).all(), "s_z has NaNs — BioVU rows leaked into train"

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate(name, pipe, X, y):
    scores = []
    for tr, te in cv.split(X):
        pipe.fit(X[tr], y[tr])
        pred = pipe.predict(X[te])
        scores.append(r2_score(y[te], pred))
    scores = np.asarray(scores)
    print(f"{name:25s} | CV R² = {scores.mean():.3f} ± {scores.std():.3f}  folds={np.round(scores,3)}")
    return scores

enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_gene = enc.fit_transform(genes)

pipe_gene = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", RidgeCV(alphas=np.logspace(-2, 3, 25))),
])
r2_gene = evaluate("Gene only", pipe_gene, X_gene, y)

pipe_esm = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=min(16, X_delta.shape[0]-5), random_state=42)),
    ("reg", RidgeCV(alphas=np.logspace(-2, 3, 25))),
])
r2_esm = evaluate("BioNeMo ESM-2 delta only", pipe_esm, X_delta, y)

X_comb = np.hstack([X_gene, X_delta])
pipe_full = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=min(24, X_comb.shape[0]-5), random_state=42)),
    ("reg", RidgeCV(alphas=np.logspace(-2, 3, 25))),
])
r2_full = evaluate("Gene + BioNeMo ESM-2", pipe_full, X_comb, y)

print("=" * 55)
print("If all three CV means are negative, stop. Do not report ΔR².")
print("Negative R² = worse than predicting the training-fold mean.")
print("=" * 55)

# Optional in-sample fit for artefact only — not a result
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=min(24, X_comb.shape[0]-5), random_state=42)),
    ("reg", RidgeCV(alphas=np.logspace(-2, 3, 25))),
])
final_pipe.fit(X_comb, y)
y_pred = final_pipe.predict(X_comb)
print(f"(in-sample, not CV) R² = {r2_score(y, y_pred):.3f}   RMSE = {np.sqrt(mean_squared_error(y, y_pred)):.4f}")

Path("models").mkdir(exist_ok=True)
joblib.dump(final_pipe, "models/fitness_regressor_bionemo.joblib")
meta = {
    "train_cohort": "LBC_SardiNIA_only",
    "n_variants": int(len(pooled_emb)),
    "target": "s_z",
    "cv_r2_gene_mean": float(r2_gene.mean()),
    "cv_r2_esm_mean": float(r2_esm.mean()),
    "cv_r2_full_mean": float(r2_full.mean()),
    "note": "BioVU excluded from train: no synonymous noise floor; raw s not commensurate with s_z.",
}
json.dump(meta, open("models/fitness_regressor_meta.json", "w"), indent=2)
print("Saved models/fitness_regressor_bionemo.joblib + meta (LBC/SardiNIA only)")


y = s_z  mean=0.621  std=1.302  n=260
Gene only                 | CV R² = -0.194 ± 0.182  folds=[-0.537 -0.218 -0.09  -0.094 -0.031]
BioNeMo ESM-2 delta only  | CV R² = -0.194 ± 0.221  folds=[-0.631 -0.142 -0.05  -0.048 -0.101]
Gene + BioNeMo ESM-2      | CV R² = -0.219 ± 0.260  folds=[-0.727 -0.198 -0.061 -0.041 -0.067]
If all three CV means are negative, stop. Do not report ΔR².
Negative R² = worse than predicting the training-fold mean.
(in-sample, not CV) R² = 0.087   RMSE = 1.2443
Saved models/fitness_regressor_bionemo.joblib + meta (LBC/SardiNIA only)


## 7. What to record after this fix

- Train n = unique `mut_key` in LBC+SardiNIA, not concatenated BioVU rows
- Overlap table: Pearson/Spearman of raw `s` on the ~53 shared keys
- BioVU is a **raw-s holdout**, not a third z-scored cohort
- CV R² only. In-sample R² is not a result
- Do not put a ΔR² on the one-pager / Hugging Face card if CV R² is negative

**Still true.** Slopes from few waves. LBC time is approximated. SardiNIA synonymous set is ascertainment-biased. VAF is not clone size if copy number changes.
